In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# =========================
# RUTAS DEL PROYECTO
# =========================

PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw"

CLIENTS_PATH = DATA_DIR / "clients_raw.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions_raw.csv"
GROUND_TRUTH_PATH = DATA_DIR / "_ground_truth.csv"

OUTPUT_SILVER_DIR = PROJECT_DIR / "outputs" / "m2" / "silver"
OUTPUT_SILVER_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EDA_DIR = PROJECT_DIR / "outputs" / "m2" / "eda"
OUTPUT_EDA_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = pd.Timestamp("2025-06-30")
ANALYSIS_WINDOW_DAYS = 180
NEW_CLIENT_WINDOW_DAYS = 180
MIN_TRANSACTIONS_FOR_CHURN = 3

snapshot_date = 2025-06-30
ventana de análisis = 180 días
cliente nuevo = menos de 180 días de historial transaccional
mínimo para evaluar churn = 3 transacciones totales

In [13]:
clients_raw = pd.read_csv(CLIENTS_PATH)
transactions_raw = pd.read_csv(TRANSACTIONS_PATH)
ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

print("Clientes:", clients_raw.shape)
print("Transacciones:", transactions_raw.shape)

Clientes: (500, 7)
Transacciones: (68483, 6)


* 500 clientes registrados
* 68,483 transacciones históricas

In [14]:
# convertir a fechas
clients = clients_raw.copy()
transactions = transactions_raw.copy()

clients["registration_date"] = pd.to_datetime(clients["registration_date"])
transactions["date"] = pd.to_datetime(transactions["date"])

transactions = transactions[transactions["date"] <= SNAPSHOT_DATE].copy()

print("Fecha mínima de transacción:", transactions["date"].min().date())
print("Fecha máxima de transacción:", transactions["date"].max().date())
print("Transacciones usadas:", transactions.shape)

Fecha mínima de transacción: 2021-01-01
Fecha máxima de transacción: 2025-06-30
Transacciones usadas: (68483, 6)


In [15]:
# Separar clientes con y sin transacciones
clients_with_transactions = clients[
    clients["client_id"].isin(transactions["client_id"])
].copy()

clients_without_transactions = clients[
    ~clients["client_id"].isin(transactions["client_id"])
].copy()

print("Clientes con transacciones:", clients_with_transactions.shape)
print("Clientes sin transacciones:", clients_without_transactions.shape)

Clientes con transacciones: (498, 7)
Clientes sin transacciones: (2, 7)


498 clientes entran al modelamiento LRFMV
2 clientes quedan fuera del modelo

In [16]:
# Calcular primera, ultima compra y cantidad de transacciones totales
client_history = (
    transactions
    .groupby("client_id")
    .agg(
        first_purchase_date=("date", "min"),
        last_purchase_date=("date", "max"),
        n_transactions_total=("transaction_id", "count"),
    )
    .reset_index()
)

client_history["length_days"] = (
    SNAPSHOT_DATE - client_history["first_purchase_date"]
).dt.days

client_history["recency_days"] = (
    SNAPSHOT_DATE - client_history["last_purchase_date"]
).dt.days

client_history.head()

,client_id,first_purchase_date,last_purchase_date,n_transactions_total,length_days,recency_days
0,CLI-0001,2021-10-13,2025-02-27,137,1356,123
1,CLI-0002,2022-02-03,2025-06-29,148,1243,1
2,CLI-0003,2022-04-03,2025-06-28,185,1184,2
3,CLI-0004,2021-12-05,2025-06-30,175,1303,0
4,CLI-0005,2021-03-15,2024-08-30,134,1568,304


* first_purchase_date = primera compra observada
* last_purchase_date = última compra observada
* n_transactions_total = número total de compras históricas
* length_days = días desde la primera compra hasta el snapshot
* recency_days = días desde la última compra hasta el snapshot

In [ ]:
# Revisando length_days, recency_days y n_transactions_total
client_history[
    ["length_days", "recency_days", "n_transactions_total"]
].describe()

,length_days,recency_days,n_transactions_total
count,498.000000,498.000000,498.000000
mean,1123.311245,90.600402,137.516064
std,520.248274,168.349303,88.305173
min,0.000000,0.000000,1.000000
25%,1027.750000,3.000000,60.500000
50%,1304.000000,11.000000,149.000000
75%,1514.000000,82.000000,196.000000
max,1641.000000,923.000000,396.000000


Hay clientes muy antiguos y clientes muy recientes.
Hay clientes que compraron el mismo día del snapshot.
Hay clientes que llevan muchos días sin comprar.
Hay clientes con una sola compra y otros con cientos.

In [18]:
# crear ventana de los ultimos 180 dias
transactions["days_since_purchase"] = (
    SNAPSHOT_DATE - transactions["date"]
).dt.days

transactions_180 = transactions[
    transactions["days_since_purchase"] <= ANALYSIS_WINDOW_DAYS
].copy()

print("Transacciones en últimos 180 días:", transactions_180.shape)
print("Fecha mínima en ventana 180 días:", transactions_180["date"].min().date())
print("Fecha máxima en ventana 180 días:", transactions_180["date"].max().date())

Transacciones en últimos 180 días: (7162, 7)
Fecha mínima en ventana 180 días: 2025-01-01
Fecha máxima en ventana 180 días: 2025-06-30


* Aquí filtramos la actividad comercial reciente.

* ¿Por qué 180 días?

Porque queremos que Frequency, Monetary y Volume representen comportamiento reciente, no toda la vida del cliente.
* No es lo mismo un cliente que compró mucho hace tres años
que un cliente que sigue comprando activamente en los últimos meses.

In [ ]:
# resumen EDA Bronze
client_duplicates = clients["client_id"].duplicated().sum()
transaction_duplicates = transactions["transaction_id"].duplicated().sum()
invalid_registration_dates = clients["registration_date"].isna().sum()
invalid_transaction_dates = transactions["date"].isna().sum()
amount_le_zero = (transactions["amount"] <= 0).sum()
sku_count_le_zero = (transactions["sku_count"] <= 0).sum()

clients_with_transactions = set(transactions["client_id"].unique()) & set(clients["client_id"].unique())
clients_without_transactions = set(clients["client_id"].unique()) - set(transactions["client_id"].unique())
transactions_without_master_client = set(transactions["client_id"].unique()) - set(clients["client_id"].unique())

eda_summary = {
    "n_clients": len(clients),
    "n_transactiones": len(transactions),
    "n_ground_truth": len(ground_truth),
    "client_id_duplicados": int(client_duplicates),
    "transactiones_id_duplicados": int(transaction_duplicates),
    "invalid_registration_dates": int(invalid_registration_dates),
    "invalid_transaction_dates": int(invalid_transaction_dates),
    "amount_le_zero": int(amount_le_zero),
    "sku_count_le_zero": int(sku_count_le_zero),
    "clients_con_transactions": len(clients_with_transactions),
    "clients_sin_transactions": len(clients_without_transactions),
    "clientes_primera_transaccion_antes_registro": len(clientes_primera_transaccion_antes_registro),
    "transactions_without_master_client": len(transactions_without_master_client),
    "transaction_date_min": str(transactions["date"].min().date()),
    "transaction_date_max": str(transactions["date"].max().date()),
    "registration_date_min": str(clients["registration_date"].min().date()),
    "registration_date_max": str(clients["registration_date"].max().date()),
}

eda_summary_path = OUTPUT_EDA_DIR / "eda_summary.json"
eda_summary_path.write_text(
    json.dumps(eda_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Resumen EDA guardado en: {eda_summary_path}")
eda_summary


Resumen EDA guardado en: ..\outputs\m2\eda\eda_summary.json


{'n_clients': 500,
 'n_transactiones': 68483,
 'n_ground_truth': 500,
 'client_id_duplicados': 0,
 'transactiones_id_duplicados': 0,
 'invalid_registration_dates': 0,
 'invalid_transaction_dates': 0,
 'amount_le_zero': 0,
 'sku_count_le_zero': 0,
 'clients_con_transactions': 498,
 'clients_sin_transactions': 2,
 'clientes_primera_transaccion_antes_registro': 340,
 'transactions_without_master_client': 0,
 'transaction_date_min': '2021-01-01',
 'transaction_date_max': '2025-06-30',
 'registration_date_min': '2021-01-01',
 'registration_date_max': '2025-06-29'}

In [19]:
# Calcular volumen por categorias
def count_distinct_categories(series):
    categories = set()

    for value in series.dropna():
        parsed_categories = [
            category.strip()
            for category in str(value).split("|")
            if category.strip()
        ]

        categories.update(parsed_categories)

    return len(categories)

In [21]:
transactions_180

,transaction_id,client_id,date,amount,sku_count,categories_purchased,days_since_purchase
61321,TXN-061322,CLI-0116,2025-01-01,122.77,7,gaseosas|agua|jugos,180
61322,TXN-061323,CLI-0412,2025-01-01,100.63,5,gaseosas|agua,180
61323,TXN-061324,CLI-0276,2025-01-01,82.27,6,gaseosas|agua,180
61324,TXN-061325,CLI-0003,2025-01-01,155.46,5,gaseosas|agua|jugos,180
61325,TXN-061326,CLI-0148,2025-01-01,80.84,5,gaseosas|agua,180
...,...,...,...,...,...,...,...
68478,TXN-068479,CLI-0115,2025-06-30,133.27,4,gaseosas|agua,0
68479,TXN-068480,CLI-0403,2025-06-30,128.16,5,gaseosas|agua|jugos,0
68480,TXN-068481,CLI-0235,2025-06-30,177.70,5,gaseosas|agua|jugos,0
68481,TXN-068482,CLI-0252,2025-06-30,180.09,4,gaseosas|agua,0


In [20]:
# Calcular Frequency, Monetary y Volume
recent_features = (
    transactions_180
    .groupby("client_id")
    .agg(
        frequency=("transaction_id", "count"),
        monetary=("amount", "sum"),
        volume=("categories_purchased", count_distinct_categories),
    )
    .reset_index()
)

recent_features.head()

,client_id,frequency,monetary,volume
0,CLI-0001,1,61.68,1
1,CLI-0002,21,3861.92,3
2,CLI-0003,17,3361.16,3
3,CLI-0004,16,2336.61,3
4,CLI-0007,4,254.52,2


frequency = número de compras en últimos 180 días
monetary = suma de ventas en últimos 180 días
volume = número de categorías distintas compradas en últimos 180 días

In [ ]:
# Unir historial total con features recientes
clients_features = (
    client_history
    .merge(recent_features, on="client_id", how="left")
)

cols_to_fill = ["frequency", "monetary", "volume"]

clients_features[cols_to_fill] = clients_features[cols_to_fill].fillna(0)

clients_features["frequency"] = clients_features["frequency"].astype(int)
clients_features["volume"] = clients_features["volume"].astype(int)

clients_features.head()

,client_id,first_purchase_date,last_purchase_date,n_transactions_total,length_days,recency_days,frequency,monetary,volume
0,CLI-0001,2021-10-13,2025-02-27,137,1356,123,1,61.68,1
1,CLI-0002,2022-02-03,2025-06-29,148,1243,1,21,3861.92,3
2,CLI-0003,2022-04-03,2025-06-28,185,1184,2,17,3361.16,3
3,CLI-0004,2021-12-05,2025-06-30,175,1303,0,16,2336.61,3
4,CLI-0005,2021-03-15,2024-08-30,134,1568,304,0,0.00,0


In [ ]:
# calcular frecuencia ultimos 90 dias y 90 dias anteriores
transactions_last_90 = transactions[
    transactions["days_since_purchase"] <= 90
].copy()

transactions_previous_90 = transactions[
    (transactions["days_since_purchase"] > 90)
    & (transactions["days_since_purchase"] <= 180)
].copy()

frequency_last_90 = (
    transactions_last_90
    .groupby("client_id")
    .size()
    .reset_index(name="frequency_last_3_months")
)

frequency_previous_90 = (
    transactions_previous_90
    .groupby("client_id")
    .size()
    .reset_index(name="frequency_previous_3_months")
)

clients_features = (
    clients_features
    .merge(frequency_last_90, on="client_id", how="left")
    .merge(frequency_previous_90, on="client_id", how="left")
)

clients_features[
    ["frequency_last_3_months", "frequency_previous_3_months"]
] = clients_features[
    ["frequency_last_3_months", "frequency_previous_3_months"]
].fillna(0).astype(int)

clients_features["delta_frequency"] = (
    clients_features["frequency_last_3_months"]
    - clients_features["frequency_previous_3_months"]
)

clients_features[
    [
        "client_id",
        "frequency",
        "frequency_last_3_months",
        "frequency_previous_3_months",
        "delta_frequency",
    ]
].head()

,client_id,frequency,frequency_last_3_months,frequency_previous_3_months,delta_frequency
0,CLI-0001,1,0,1,-1
1,CLI-0002,21,11,10,1
2,CLI-0003,17,9,8,1
3,CLI-0004,16,7,9,-2
4,CLI-0005,0,0,0,0


In [26]:
# revisar distribuciones delta_frequency
clients_features[
    [
        "frequency_last_3_months",
        "frequency_previous_3_months",
        "delta_frequency",
    ]
].describe()

,frequency_last_3_months,frequency_previous_3_months,delta_frequency
count,498.000000,498.000000,498.000000
mean,7.351406,7.030120,0.321285
std,6.750438,6.719824,4.937557
min,0.000000,0.000000,-18.000000
25%,1.000000,0.000000,-2.000000
50%,6.000000,6.000000,0.000000
75%,12.000000,12.000000,3.000000
max,29.000000,32.000000,19.000000


* La media de delta_frequency está cerca de cero.

* Eso significa que, en promedio, la cartera está relativamente estable, pero hay clientes con: 

            * fuertes caídas de frecuencia
            * fuertes mejoras de frecuencia
* Los clientes con caída serán importantes para churn.

In [27]:
#calcular es_nuevo y chrun_elegible
clients_features["es_nuevo"] = (
    clients_features["length_days"] < NEW_CLIENT_WINDOW_DAYS
)

clients_features["churn_eligible"] = (
    (clients_features["length_days"] >= NEW_CLIENT_WINDOW_DAYS)
    & (clients_features["n_transactions_total"] >= MIN_TRANSACTIONS_FOR_CHURN)
)

print("Clientes nuevos:", clients_features["es_nuevo"].sum())
print("Clientes elegibles para churn:", clients_features["churn_eligible"].sum())

Clientes nuevos: 84
Clientes elegibles para churn: 414


No evaluamos churn en clientes nuevos porque podrían tener baja frecuencia simplemente por falta de historial, no necesariamente por abandono.

In [28]:
# unir datos descriptivos del cliente
clients_features = (
    clients_with_transactions
    .merge(clients_features, on="client_id", how="inner")
)

print("Shape final Silver antes del score:", clients_features.shape)

clients_features.head()

Shape final Silver antes del score: (498, 20)


,client_id,business_name,store_type,zone,latitude,longitude,registration_date,first_purchase_date,last_purchase_date,n_transactions_total,length_days,recency_days,frequency,monetary,volume,frequency_last_3_months,frequency_previous_3_months,delta_frequency,es_nuevo,churn_eligible
0,CLI-0001,Pollería El Centro,restaurante,Chilca,-12.082178,-75.200853,2021-10-14,2021-10-13,2025-02-27,137,1356,123,1,61.68,1,0,1,-1,False,True
1,CLI-0002,Tienda Wanka,bodega,Chilca,-12.082362,-75.192667,2022-02-19,2022-02-03,2025-06-29,148,1243,1,21,3861.92,3,11,10,1,False,True
2,CLI-0003,Tienda Santa Rosa,bodega,San Carlos,-12.053894,-75.227877,2022-04-13,2022-04-03,2025-06-28,185,1184,2,17,3361.16,3,9,8,1,False,True
3,CLI-0004,Licores San Martín,licoreria,Chilca,-12.085464,-75.196285,2021-12-11,2021-12-05,2025-06-30,175,1303,0,16,2336.61,3,7,9,-2,False,True
4,CLI-0005,Tienda El Progreso,bodega,El Tambo,-12.043708,-75.208575,2021-03-07,2021-03-15,2024-08-30,134,1568,304,0,0.00,0,0,0,0,False,True


In [29]:
# Aplicar log1p a Recency y Monetary
clients_features["R_log"] = np.log1p(clients_features["recency_days"])
clients_features["M_log"] = np.log1p(clients_features["monetary"])

clients_features[
    ["recency_days", "R_log", "monetary", "M_log"]
].head()

,recency_days,R_log,monetary,M_log
0,123,4.820282,61.68,4.138042
1,1,0.693147,3861.92,8.259179
2,2,1.098612,3361.16,8.120339
3,0,0.000000,2336.61,7.756884
4,304,5.720312,0.00,0.000000


Aplicamos log1p a:

        * recency_days
        * monetary

¿Por qué?

Porque suelen tener cola positiva:

* algunos clientes llevan muchísimo tiempo sin comprar
* algunos clientes concentran montos muy altos

log1p reduce esa asimetría sin destruir el orden de los datos.

In [30]:
# Revisar asimetría antes del score
clients_features[
    ["length_days", "recency_days", "frequency", "monetary", "volume"]
].skew()

length_days    -1.175488
recency_days    2.427353
frequency       0.738491
monetary        2.328743
volume         -0.635046
dtype: float64

recency_days tiene fuerte cola positiva
monetary tiene fuerte cola positiva

Por eso usamos:

        * R_log
        * M_log

En cambio:

frequency se mantiene cruda
volume se mantiene cruda
length_days se mantiene cruda

In [31]:
# Escalar variables con MinMaxScaler
score_variables = [
    "length_days",
    "R_log",
    "frequency",
    "M_log",
    "volume",
]

minmax_scaler = MinMaxScaler()

scaled_values = minmax_scaler.fit_transform(
    clients_features[score_variables]
)

scaled_features = pd.DataFrame(
    scaled_values,
    columns=[
        "L_minmax",
        "R_minmax",
        "F_minmax",
        "M_minmax",
        "V_minmax",
    ],
    index=clients_features.index,
)

clients_features = pd.concat(
    [clients_features, scaled_features],
    axis=1,
)

clients_features[
    ["L_minmax", "R_minmax", "F_minmax", "M_minmax", "V_minmax"]
].describe()

,L_minmax,R_minmax,F_minmax,M_minmax,V_minmax
count,498.000000,498.000000,498.000000,498.000000,498.000000
mean,0.684528,0.424122,0.256813,0.600948,0.544679
std,0.317031,0.270693,0.223799,0.309769,0.308150
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.626295,0.203010,0.053571,0.552592,0.500000
50%,0.794637,0.363891,0.232143,0.723726,0.500000
75%,0.922608,0.647065,0.410714,0.815237,0.750000
max,1.000000,1.000000,1.000000,1.000000,1.000000


MinMaxScaler transforma cada variable a escala:

0 a 1

Esto sirve porque el score final debe ser interpretable:

0 a 100

No usamos StandardScaler para score porque un score con valores negativos o desviaciones estándar no sería tan fácil de explicar a negocio.

In [32]:
# invertir recency
clients_features["R_score"] = 1 - clients_features["R_minmax"]

clients_features[
    ["recency_days", "R_minmax", "R_score"]
].sort_values("recency_days").head()

,recency_days,R_minmax,R_score
24,0,0.0,1.0
414,0,0.0,1.0
400,0,0.0,1.0
377,0,0.0,1.0
368,0,0.0,1.0


Recency es especial.

En casi todas las variables:

más alto = mejor

Pero en Recency:

menos días desde la última compra = mejor

Por eso invertimos:

R_score = 1 - R_minmax

Entonces:

cliente que compró recientemente → R_score alto
cliente que lleva mucho sin comprar → R_score bajo

In [33]:
# Calcular score lrfmv 0-100
clients_features["score_lrfmv"] = (
    0.30 * clients_features["F_minmax"]
    + 0.20 * clients_features["R_score"]
    + 0.20 * clients_features["M_minmax"]
    + 0.20 * clients_features["V_minmax"]
    + 0.10 * clients_features["L_minmax"]
)

clients_features["score_lrfmv_0_100"] = (
    clients_features["score_lrfmv"] * 100
)

clients_features[
    [
        "client_id",
        "frequency",
        "recency_days",
        "monetary",
        "volume",
        "length_days",
        "score_lrfmv_0_100",
    ]
].sort_values("score_lrfmv_0_100", ascending=False).head(10)

,client_id,frequency,recency_days,monetary,volume,length_days,score_lrfmv_0_100
416,CLI-0419,56,2,14022.98,4,1489,95.141861
199,CLI-0200,56,5,19972.34,4,1549,94.191643
175,CLI-0176,48,1,16761.02,4,1581,92.964529
288,CLI-0289,56,1,10450.68,3,1607,91.454643
418,CLI-0421,48,3,17454.70,4,1548,90.815238
362,CLI-0365,39,0,8631.06,4,1581,88.832817
372,CLI-0375,54,2,9314.51,3,1483,88.207609
121,CLI-0122,42,3,12858.55,4,1582,87.190945
495,CLI-0498,45,6,8792.60,4,1547,86.178159
59,CLI-0060,37,1,12701.64,4,1520,86.139856


El score combina:

Frequency  → 30%
Recency    → 20%
Monetary   → 20%
Volume     → 20%
Length     → 10%

Lectura de negocio:

score alto = cliente prioritario comercialmente
score bajo = cliente de menor actividad o menor valor reciente

In [34]:
# Revisar distribución del score
clients_features["score_lrfmv_0_100"].describe()

count    498.000000
mean      48.979761
std       23.118943
min        7.330684
25%       35.409505
50%       52.738036
75%       67.920502
max       95.141861
Name: score_lrfmv_0_100, dtype: float64

El cliente promedio tiene score cercano a 49.
La mediana está cerca de 53.
Los mejores clientes llegan a scores cercanos a 95.
Los más débiles están cerca de 7.

In [35]:
# Vista final de Silver
silver_columns = [
    "client_id",
    "business_name",
    "store_type",
    "zone",
    "registration_date",
    "first_purchase_date",
    "last_purchase_date",
    "length_days",
    "recency_days",
    "frequency",
    "monetary",
    "volume",
    "n_transactions_total",
    "frequency_last_3_months",
    "frequency_previous_3_months",
    "delta_frequency",
    "es_nuevo",
    "churn_eligible",
    "R_log",
    "M_log",
    "L_minmax",
    "R_minmax",
    "R_score",
    "F_minmax",
    "M_minmax",
    "V_minmax",
    "score_lrfmv",
    "score_lrfmv_0_100",
]

clients_features = clients_features[silver_columns].copy()

clients_features.head()

,client_id,business_name,store_type,zone,registration_date,first_purchase_date,last_purchase_date,length_days,recency_days,frequency,...,R_log,M_log,L_minmax,R_minmax,R_score,F_minmax,M_minmax,V_minmax,score_lrfmv,score_lrfmv_0_100
0,CLI-0001,Pollería El Centro,restaurante,Chilca,2021-10-14,2021-10-13,2025-02-27,1356,123,1,...,4.820282,4.138042,0.826325,0.705884,0.294116,0.017857,0.417893,0.25,0.280391,28.039143
1,CLI-0002,Tienda Wanka,bodega,Chilca,2022-02-19,2022-02-03,2025-06-29,1243,1,21,...,0.693147,8.259179,0.757465,0.101505,0.898495,0.375000,0.834079,0.75,0.684761,68.476134
2,CLI-0003,Tienda Santa Rosa,bodega,San Carlos,2022-04-13,2022-04-03,2025-06-28,1184,2,17,...,1.098612,8.120339,0.721511,0.160881,0.839119,0.303571,0.820058,0.75,0.645058,64.505786
3,CLI-0004,Licores San Martín,licoreria,Chilca,2021-12-11,2021-12-05,2025-06-30,1303,0,16,...,0.000000,7.756884,0.794028,0.000000,1.000000,0.285714,0.783353,0.75,0.671788,67.178774
4,CLI-0005,Tienda El Progreso,bodega,El Tambo,2021-03-07,2021-03-15,2024-08-30,1568,304,0,...,5.720312,0.000000,0.955515,0.837685,0.162315,0.000000,0.000000,0.00,0.128014,12.801443


In [36]:
# Exportar Silver
clients_features.to_csv(
    OUTPUT_SILVER_DIR / "clients_features.csv",
    index=False,
)

clients_without_transactions.to_csv(
    OUTPUT_SILVER_DIR / "clients_without_transactions.csv",
    index=False,
)

print("Silver exportado correctamente.")
print("Archivo:", OUTPUT_SILVER_DIR / "clients_features.csv")
print("Filas:", clients_features.shape[0])
print("Columnas:", clients_features.shape[1])

Silver exportado correctamente.
Archivo: E:\proyecto_2026\2026-II\project_bebidas\modules\m2_lrfmv\outputs\m2\silver\clients_features.csv
Filas: 498
Columnas: 28
